In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from ipywidgets import interact, IntSlider, Dropdown, HBox, VBox
import ipywidgets as widgets
%matplotlib inline
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from scipy.ndimage import zoom as spzoom

In [ ]:
NUM_PATH  = '/cephfs/store/gr-mc2473/ham51/realpdebench/well/cylinder_numerical.hdf5'
REAL_PATH = '/cephfs/store/gr-mc2473/ham51/realpdebench/well/cylinder_real.hdf5'

# Read Reynolds numbers and n_trajectories from both files
with h5py.File(NUM_PATH, 'r') as f:
    re_num  = f.attrs['reynolds_number']  # shape (92,)
    n_num   = int(f.attrs['n_trajectories'])

with h5py.File(REAL_PATH, 'r') as f:
    re_real = f.attrs['reynolds_number']
    n_real  = int(f.attrs['n_trajectories'])

print(f"Numerical: {n_num} trajectories, Re range [{re_num.min():.0f}, {re_num.max():.0f}]")
print(f"Real:      {n_real} trajectories, Re range [{re_real.min():.0f}, {re_real.max():.0f}]")

In [ ]:
# ── Load one trajectory from each dataset into memory ──────────────────────
# Change these indices to pick a different trajectory.
# Tip: find matching Re numbers with np.argmin(np.abs(re_num - re_real[i]))

NUM_TRAJ_IDX  = 0   # index into numerical dataset
REAL_TRAJ_IDX = 0   # index into real dataset

print(f"Loading numerical trajectory {NUM_TRAJ_IDX}  (Re={re_num[NUM_TRAJ_IDX]:.0f}) ...")
with h5py.File(NUM_PATH, 'r') as f:
    vel_num  = f['t1_fields']['velocity'][NUM_TRAJ_IDX]   # (T, W, H, 2)
    pres_num = f['t0_fields']['pressure'][NUM_TRAJ_IDX]   # (T, W, H)
    t_axis   = f['dimensions']['time'][:]                 # (T,) physical time in seconds

print(f"Loading real trajectory      {REAL_TRAJ_IDX}  (Re={re_real[REAL_TRAJ_IDX]:.0f}) ...")
with h5py.File(REAL_PATH, 'r') as f:
    vel_real  = f['t1_fields']['velocity'][REAL_TRAJ_IDX]   # (T, W, H, 2)
    vort_real = f['t0_fields']['vorticity'][REAL_TRAJ_IDX]  # (T, W, H)

T_num  = vel_num.shape[0]
T_real = vel_real.shape[0]
T_max  = min(T_num, T_real)

dt_ms = np.diff(t_axis).mean() * 1000
print(f"Numerical shape: {vel_num.shape}  (T={T_num}, W={vel_num.shape[1]}, H={vel_num.shape[2]})")
print(f"Real shape:      {vel_real.shape}  (T={T_real}, W={vel_real.shape[1]}, H={vel_real.shape[2]})")
print(f"Physical time:   {t_axis[0]:.4f}s → {t_axis[-1]:.4f}s  (dt = {dt_ms:.2f} ms/frame)")

In [ ]:
# ── Helper: (T, W, H) → imshow-ready (H, W) slice at time t ───────────────
def frame(arr, t):
    """Return arr[t] transposed from (W, H) to (H, W) for imshow."""
    return arr[t].T

# Pre-compute symmetric colourbar limits from a sample of frames
def sym_lim(arr, pct=99):
    v = np.percentile(np.abs(arr[::50]), pct)
    return -v, v

clim_u_num   = sym_lim(vel_num[..., 0])
clim_v_num   = sym_lim(vel_num[..., 1])
clim_p       = sym_lim(pres_num)
clim_u_real  = sym_lim(vel_real[..., 0])
clim_v_real  = sym_lim(vel_real[..., 1])
clim_vort    = sym_lim(vort_real)

In [ ]:
# ── Interactive explorer ───────────────────────────────────────────────────
STEP = 10

def plot_frame(t):
    fig, axes = plt.subplots(2, 2, figsize=(12, 7))
    fig.subplots_adjust(left=0.05, right=0.97, hspace=0.35, wspace=0.30)
    fig.suptitle(
        f'Real Re={re_real[REAL_TRAJ_IDX]:.0f}   Sim Re={re_num[NUM_TRAJ_IDX]:.0f}'
        f'     t={t}  ({t_axis[t]:.3f} s)',
        fontsize=11
    )
    panels = [
        (axes[0,0], 'Real  u', vel_real[t,:,:,0].T, clim_u_real),
        (axes[0,1], 'Sim   u', vel_num [t,:,:,0].T, clim_u_num),
        (axes[1,0], 'Real  v', vel_real[t,:,:,1].T, clim_v_real),
        (axes[1,1], 'Sim   v', vel_num [t,:,:,1].T, clim_v_num),
    ]
    for ax, title, data, clim in panels:
        im = ax.imshow(data, origin='lower', aspect='auto',
                       cmap='RdBu_r', vmin=clim[0], vmax=clim[1])
        ax.set_title(title, fontsize=10)
        ax.axis('off')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.show()

slider = IntSlider(min=0, max=T_max - 1, step=STEP, value=0,
                   description='t =', continuous_update=False,
                   layout=widgets.Layout(width='80%'))

interact(plot_frame, t=slider);

In [ ]:
# ── Option A: same plot_frame, but with a ▶ play button ───────────────────
play = widgets.Play(
    min=0, max=T_max - 1, step=STEP,
    interval=500,   # ms between frames — raise to 800+ if rendering lags
    value=0, loop=True,
)
play_slider = IntSlider(min=0, max=T_max - 1, step=STEP, value=0,
                        description='t =', continuous_update=True,
                        layout=widgets.Layout(width='70%'))
widgets.jslink((play, 'value'), (play_slider, 'value'))
out = widgets.interactive_output(plot_frame, {'t': play_slider})
display(HBox([play, play_slider]), out)

In [ ]:
# ── Option B: FuncAnimation → embedded HTML5 video ────────────────────────
import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 200   # MB

T_VID    = min(1000, T_max)   # first ~5s of physics
STEP_VID = 10                 # 100 frames total, renders in ~20s
frames_t = list(range(0, T_VID, STEP_VID))
n_frames = len(frames_t)

print(f"{n_frames} frames  |  t=0..{frames_t[-1]}  ({t_axis[0]:.3f}s → {t_axis[frames_t[-1]]:.3f}s,  step={STEP_VID * np.diff(t_axis).mean()*1000:.0f} ms)")

fig_v, axes = plt.subplots(2, 2, figsize=(12, 7))
fig_v.subplots_adjust(left=0.05, right=0.97, hspace=0.35, wspace=0.30)

t0 = frames_t[0]
kw = dict(origin='lower', aspect='auto', cmap='RdBu_r')
ims = [
    axes[0,0].imshow(vel_real[t0,:,:,0].T, vmin=clim_u_real[0], vmax=clim_u_real[1], **kw),
    axes[0,1].imshow(vel_num [t0,:,:,0].T, vmin=clim_u_num[0],  vmax=clim_u_num[1],  **kw),
    axes[1,0].imshow(vel_real[t0,:,:,1].T, vmin=clim_v_real[0], vmax=clim_v_real[1], **kw),
    axes[1,1].imshow(vel_num [t0,:,:,1].T, vmin=clim_v_num[0],  vmax=clim_v_num[1],  **kw),
]
for im, ax, lbl in zip(ims, axes.flat, ['Real  u', 'Sim  u', 'Real  v', 'Sim  v']):
    ax.set_title(lbl, fontsize=10)
    ax.axis('off')
    fig_v.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

def _update(i):
    t = frames_t[i]
    ims[0].set_data(vel_real[t,:,:,0].T)
    ims[1].set_data(vel_num [t,:,:,0].T)
    ims[2].set_data(vel_real[t,:,:,1].T)
    ims[3].set_data(vel_num [t,:,:,1].T)
    fig_v.suptitle(
        f'Real Re={re_real[REAL_TRAJ_IDX]:.0f}   Sim Re={re_num[NUM_TRAJ_IDX]:.0f}'
        f'     t={t}  ({t_axis[t]:.3f}s)',
        fontsize=11
    )

print(f"Rendering {n_frames} frames …")
anim = FuncAnimation(fig_v, _update, frames=n_frames, interval=80, blit=False)
plt.close(fig_v)
HTML(anim.to_jshtml())

In [ ]:
# ── Generate + save HTML videos for low / mid / high Re ───────────────────
import matplotlib as mpl
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
mpl.rcParams['animation.embed_limit'] = 200

OUT_DIR  = '/cephfs/store/gr-mc2473/ham51/realpdebench/notebooks'
T_VID    = min(1000, T_max)
STEP_VID = 10
frames_t = list(range(0, T_VID, STEP_VID))
n_frames = len(frames_t)

RE_TARGETS = {'low': 2500, 'mid': 6000, 'high': 10500}

for label, re_target in RE_TARGETS.items():
    idx = int(np.argmin(np.abs(re_num - re_target)))
    re_val = re_num[idx]
    print(f"\n── {label}  (Re={re_val:.0f},  traj idx={idx}) ──")

    with h5py.File(NUM_PATH, 'r') as f:
        vn = f['t1_fields']['velocity'][idx]   # (T, W, H, 2)
    with h5py.File(REAL_PATH, 'r') as f:
        vr = f['t1_fields']['velocity'][idx]   # (T, W, H, 2)

    cu = sym_lim(vn[..., 0])
    cv = sym_lim(vn[..., 1])
    cr_u = sym_lim(vr[..., 0])
    cr_v = sym_lim(vr[..., 1])

    fig, axes = plt.subplots(2, 2, figsize=(12, 7))
    fig.subplots_adjust(left=0.05, right=0.97, hspace=0.35, wspace=0.30)

    t0 = frames_t[0]
    kw = dict(origin='lower', aspect='auto', cmap='RdBu_r')
    ims = [
        axes[0,0].imshow(vr[t0,:,:,0].T, vmin=cr_u[0], vmax=cr_u[1], **kw),
        axes[0,1].imshow(vn[t0,:,:,0].T, vmin=cu[0],   vmax=cu[1],   **kw),
        axes[1,0].imshow(vr[t0,:,:,1].T, vmin=cr_v[0], vmax=cr_v[1], **kw),
        axes[1,1].imshow(vn[t0,:,:,1].T, vmin=cv[0],   vmax=cv[1],   **kw),
    ]
    for im, ax, lbl in zip(ims, axes.flat, ['Real u', 'Sim u', 'Real v', 'Sim v']):
        ax.set_title(lbl, fontsize=10)
        ax.axis('off')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    def _make_update(ims_, vr_, vn_, frames_t_):
        def _update(i):
            t = frames_t_[i]
            ims_[0].set_data(vr_[t,:,:,0].T)
            ims_[1].set_data(vn_[t,:,:,0].T)
            ims_[2].set_data(vr_[t,:,:,1].T)
            ims_[3].set_data(vn_[t,:,:,1].T)
            fig.suptitle(
                f'Re={re_val:.0f} ({label})     t={t}  ({t_axis[t]:.3f}s)',
                fontsize=11
            )
        return _update

    anim = FuncAnimation(fig, _make_update(ims, vr, vn, frames_t),
                         frames=n_frames, interval=80, blit=False)
    plt.close(fig)

    out_path = f'{OUT_DIR}/cylinder_{label}_re{re_val:.0f}.html'
    print(f"  Rendering {n_frames} frames …", end=' ', flush=True)
    html_str = anim.to_jshtml()
    with open(out_path, 'w') as f:
        f.write(html_str)
    print(f"saved → {out_path}  ({len(html_str)//1024} KB)")
    display(HTML(html_str))